In [548]:
from dataclasses import dataclass
import numpy as np

@dataclass
class Grid:
    rows: int
    cols: int
    step_reward: int
    terminals: dict
    walls: set
    actions = dict(up=(-1, 0), right=(0, 1), down=(1, 0), left=(0, -1))
    arrows = {'up': "↑", 'right': "→", 'down': "↓", 'left': "←"}
    def cell_repr(self, r, c):
        cell = (r, c)
        if cell in self.terminals:
            return self.terminals[cell]
        elif cell in self.walls:
            return '#'
        else:
            return '·'
    def render(self):
        for r in range(self.rows):
            for c in range(self.cols):
                print(f'{self.cell_repr(r, c):>3} ', end="")
            print()

    def step(self, cell, action):
        movement = self.actions[action]
        next_cell = (cell[0] + movement[0], cell[1] + movement[1])
        outside = not (0 <= next_cell[0] < self.rows and 0 <= next_cell[1] < self.cols)
        on_wall = next_cell in self.walls
        if outside or on_wall:
            next_cell = cell

        if next_cell in self.terminals:
            reward = self.terminals[next_cell]
        else:
            reward = self.step_reward
        return next_cell, reward
    def properties(self):
        return f'''
Grid(
    rows={self.rows},
    cols={self.cols},
    step_reward={self.step_reward},
    terminals={self.terminals},
    walls={self.walls},
)
'''


default_grid = Grid(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1},
    walls={(1, 1)},
)
default_grid

Grid(rows=3, cols=4, step_reward=0, terminals={(0, 3): 1}, walls={(1, 1)})

In [549]:
def render_policy(grid: Grid, policy):
    for r in range(len(policy)):
        for c in range(len(policy[0])):
            if r == 0 and c == 0:
                print("r/c", end="")
                print(''.join([f'{v:>2} ' for v in range(grid.cols)]))
            if c == 0:
                print(f'{r:>2} ', end="")

            if policy[r][c] != None:
                value = grid.arrows[max(policy[r][c], key=policy[r][c].get)]
            else:
                value = grid.cell_repr(r, c)
            print(f' {value} ', end='')
        print()
    print('------------------')

def show_V(grid: Grid, V):
    for r in range(grid.rows):
        for c in range(grid.cols):
            if r == 0 and c == 0:
                print("  r/c", end="")
                print(''.join([f'{v:>4} ' for v in range(grid.cols)]))
            if c == 0:
                print(f'{r:>4} ', end="")
            print(f'{round(V[r][c], 2):>4} ', end="")
        print()
    print('------------------')


def value_iteration(grid: Grid, gamma=0.9, theta=1e-6, max_iters=1000):
    print('---------- value_iteration ------------')
    V = [[0 for _ in range(grid.cols)] for _ in range(grid.rows)]
    policy = [[None for _ in range(grid.cols)] for _ in range(grid.rows)]
    delta = float('inf')
    i = 0
    while delta > theta and i < max_iters:
        delta = 0.0
        V_old = [row[:] for row in V]
        for r in range(grid.rows):
            for c in range(grid.cols):
                cell = (r, c)
                if cell in grid.walls or cell in grid.terminals:
                    continue
                def q(action):
                    next_cell, reward = grid.step(cell, action)
                    return reward + gamma * V_old[next_cell[0]][next_cell[1]]
                new_value = float('-inf')
                for action in grid.actions:
                    value = q(action)
                    if value > new_value:
                        new_value = value
                        policy[r][c] = {action: 1.0}
                V[r][c] = new_value
                delta = max(delta, abs(V[r][c] - V_old[r][c]))
        show_V(grid, V)
        i += 1
    render_policy(grid, policy)
    converged = delta <= theta
    if converged:
        print(f'value_iteration converged in {i} iterations')
    else:
        print(f'value_iteration did not converge in {max_iters} iterations')
    return policy, V, converged

In [550]:
policy, V, converged = value_iteration(default_grid);

---------- value_iteration ------------
  r/c   0    1    2    3 
   0  0.0  0.0  1.0    0 
   1  0.0    0  0.0  1.0 
   2  0.0  0.0  0.0  0.0 
------------------
  r/c   0    1    2    3 
   0  0.0  0.9  1.0    0 
   1  0.0    0  0.9  1.0 
   2  0.0  0.0  0.0  0.9 
------------------
  r/c   0    1    2    3 
   0 0.81  0.9  1.0    0 
   1  0.0    0  0.9  1.0 
   2  0.0  0.0 0.81  0.9 
------------------
  r/c   0    1    2    3 
   0 0.81  0.9  1.0    0 
   1 0.73    0  0.9  1.0 
   2  0.0 0.73 0.81  0.9 
------------------
  r/c   0    1    2    3 
   0 0.81  0.9  1.0    0 
   1 0.73    0  0.9  1.0 
   2 0.66 0.73 0.81  0.9 
------------------
  r/c   0    1    2    3 
   0 0.81  0.9  1.0    0 
   1 0.73    0  0.9  1.0 
   2 0.66 0.73 0.81  0.9 
------------------
r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  ↑  ↑ 
 2  ↑  →  ↑  ↑ 
------------------
value_iteration converged in 6 iterations


In [551]:
def read_policy(grid: Grid, V, gamma=0.9, incumbent_policy=None):
    policy = [[None for _ in range(grid.cols)] for _ in range(grid.rows)]
    for r in range(grid.rows):
        for c in range(grid.cols):
            cell = (r, c)
            if cell in grid.walls or cell in grid.terminals:
                continue
            def q(action):
                next_cell, reward = grid.step(cell, action)
                return reward + gamma * V[next_cell[0]][next_cell[1]]
            new_action = max(grid.actions, key=q)
            if incumbent_policy:
                incumbent_action = max(incumbent_policy[r][c], key=incumbent_policy[r][c].get)
                if q(new_action) - q(incumbent_action) < 1e-9:
                    new_action = incumbent_action
            policy[r][c] = {new_action: 1.0}
    return policy


policy = read_policy(default_grid, V)
policy

[[{'right': 1.0}, {'right': 1.0}, {'right': 1.0}, None],
 [{'up': 1.0}, None, {'up': 1.0}, {'up': 1.0}],
 [{'up': 1.0}, {'right': 1.0}, {'up': 1.0}, {'up': 1.0}]]

In [552]:
render_policy(default_grid, policy)

r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  ↑  ↑ 
 2  ↑  →  ↑  ↑ 
------------------


In [553]:
def policy_evaluation(
    grid: Grid, policy, gamma=0.9, theta=1e-6, verbose=False, max_iters=1000
):
    if verbose:
        print('---------- policy_evaluation ------------')
    V = [[0 for _ in range(grid.cols)] for _ in range(grid.rows)]
    delta = float('inf')
    i = 0
    while delta > theta and i < max_iters:
        delta = 0.0
        V_old = [row[:] for row in V]
        for r in range(grid.rows):
            for c in range(grid.cols):
                cell = (r, c)
                if cell in grid.walls or cell in grid.terminals:
                    continue
                value = 0
                for action, prob in policy[r][c].items():
                    next_cell, reward = grid.step(cell, action)
                    value += prob * (reward + gamma * V_old[next_cell[0]][next_cell[1]])
                V[r][c] = value
                delta = max(delta, abs(V[r][c] - V_old[r][c]))
        if verbose:
            show_V(grid, V)
        i += 1
    converged = delta <= theta
    if converged:
        print(f'policy_evaluation converged in {i} iterations')
    else:
        print(f'policy_evaluation did not converge in {max_iters} iterations')
    return V, converged


policy_evaluation(default_grid, policy, verbose=True);

---------- policy_evaluation ------------
  r/c   0    1    2    3 
   0  0.0  0.0  1.0    0 
   1  0.0    0  0.0  1.0 
   2  0.0  0.0  0.0  0.0 
------------------
  r/c   0    1    2    3 
   0  0.0  0.9  1.0    0 
   1  0.0    0  0.9  1.0 
   2  0.0  0.0  0.0  0.9 
------------------
  r/c   0    1    2    3 
   0 0.81  0.9  1.0    0 
   1  0.0    0  0.9  1.0 
   2  0.0  0.0 0.81  0.9 
------------------
  r/c   0    1    2    3 
   0 0.81  0.9  1.0    0 
   1 0.73    0  0.9  1.0 
   2  0.0 0.73 0.81  0.9 
------------------
  r/c   0    1    2    3 
   0 0.81  0.9  1.0    0 
   1 0.73    0  0.9  1.0 
   2 0.66 0.73 0.81  0.9 
------------------
  r/c   0    1    2    3 
   0 0.81  0.9  1.0    0 
   1 0.73    0  0.9  1.0 
   2 0.66 0.73 0.81  0.9 
------------------
policy_evaluation converged in 6 iterations


In [554]:
def policy_iteration(
    grid: Grid,
    policy=None,
    max_iters=1000,
    pass_incumbent_policy=False,
    gamma=0.9,
    theta=1e-6,
    policy_evaluation_verbose=False,
    policy_evaluation_max_iters=1000,
):
    print('---------- policy_iteration ------------')
    if policy is None:
        policy = [[{'up': 1.0} for _ in range(grid.cols)] for _ in range(grid.rows)]
    i = 0
    policy_evaluation_converged = True
    changed = None
    V = [[0 for _ in range(grid.cols)] for _ in range(grid.rows)]
    while changed != 0 and i < max_iters:
        V, policy_evaluation_converged = policy_evaluation(
            grid,
            policy,
            gamma,
            theta,
            verbose=policy_evaluation_verbose,
            max_iters=policy_evaluation_max_iters,
        )
        if not policy_evaluation_converged:
            break
        show_V(grid, V)
        new_policy = read_policy(
            grid, V, gamma, incumbent_policy=policy if pass_incumbent_policy else None
        )
        changed = sum(
            new_policy[r][c] != None
            and (
                max(new_policy[r][c], key=new_policy[r][c].get)
                != max(policy[r][c], key=policy[r][c].get)
            )
            for r in range(grid.rows)
            for c in range(grid.cols)
        )
        print(f'actions changed = {changed}')
        render_policy(grid, new_policy)
        policy = new_policy
        i += 1
    converged = changed == 0
    if not policy_evaluation_converged:
        print(
            f"policy_iteration did not converge because policy_evaluation did not converge"
        )
    elif converged:
        print(f'policy_iteration converged in {i} iterations')
    else:
        print(f'policy_iteration did not converge in {max_iters} iterations')
    return policy, V, converged

In [555]:
policy_iteration(default_grid);

---------- policy_iteration ------------
policy_evaluation converged in 3 iterations
  r/c   0    1    2    3 
   0  0.0  0.0  0.0    0 
   1  0.0    0  0.0  1.0 
   2  0.0  0.0  0.0  0.9 
------------------
actions changed = 3
r/c 0  1  2  3 
 0  ↑  ↑  →  1 
 1  ↑  #  →  ↑ 
 2  ↑  ↑  →  ↑ 
------------------
policy_evaluation converged in 4 iterations
  r/c   0    1    2    3 
   0  0.0  0.0  1.0    0 
   1  0.0    0  0.9  1.0 
   2  0.0  0.0 0.81  0.9 
------------------
actions changed = 4
r/c 0  1  2  3 
 0  ↑  →  →  1 
 1  ↑  #  ↑  ↑ 
 2  ↑  →  ↑  ↑ 
------------------
policy_evaluation converged in 5 iterations
  r/c   0    1    2    3 
   0  0.0  0.9  1.0    0 
   1  0.0    0  0.9  1.0 
   2  0.0 0.73 0.81  0.9 
------------------
actions changed = 2
r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  ↑  ↑ 
 2  →  →  ↑  ↑ 
------------------
policy_evaluation converged in 6 iterations
  r/c   0    1    2    3 
   0 0.81  0.9  1.0    0 
   1 0.73    0  0.9  1.0 
   2 0.66 0.73 0.81  0.9 
-

In [556]:
policy_iteration(default_grid, pass_incumbent_policy=True);

---------- policy_iteration ------------
policy_evaluation converged in 3 iterations
  r/c   0    1    2    3 
   0  0.0  0.0  0.0    0 
   1  0.0    0  0.0  1.0 
   2  0.0  0.0  0.0  0.9 
------------------
actions changed = 3
r/c 0  1  2  3 
 0  ↑  ↑  →  1 
 1  ↑  #  →  ↑ 
 2  ↑  ↑  →  ↑ 
------------------
policy_evaluation converged in 4 iterations
  r/c   0    1    2    3 
   0  0.0  0.0  1.0    0 
   1  0.0    0  0.9  1.0 
   2  0.0  0.0 0.81  0.9 
------------------
actions changed = 2
r/c 0  1  2  3 
 0  ↑  →  →  1 
 1  ↑  #  →  ↑ 
 2  ↑  →  →  ↑ 
------------------
policy_evaluation converged in 5 iterations
  r/c   0    1    2    3 
   0  0.0  0.9  1.0    0 
   1  0.0    0  0.9  1.0 
   2  0.0 0.73 0.81  0.9 
------------------
actions changed = 2
r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  →  ↑ 
 2  →  →  →  ↑ 
------------------
policy_evaluation converged in 6 iterations
  r/c   0    1    2    3 
   0 0.81  0.9  1.0    0 
   1 0.73    0  0.9  1.0 
   2 0.66 0.73 0.81  0.9 
-

In [557]:
uniform_policy = [
    [
        {a: 1 / len(default_grid.actions) for a in default_grid.actions}
        for _ in range(default_grid.cols)
    ]
    for _ in range(default_grid.rows)
]
uniform_policy

[[{'up': 0.25, 'right': 0.25, 'down': 0.25, 'left': 0.25},
  {'up': 0.25, 'right': 0.25, 'down': 0.25, 'left': 0.25},
  {'up': 0.25, 'right': 0.25, 'down': 0.25, 'left': 0.25},
  {'up': 0.25, 'right': 0.25, 'down': 0.25, 'left': 0.25}],
 [{'up': 0.25, 'right': 0.25, 'down': 0.25, 'left': 0.25},
  {'up': 0.25, 'right': 0.25, 'down': 0.25, 'left': 0.25},
  {'up': 0.25, 'right': 0.25, 'down': 0.25, 'left': 0.25},
  {'up': 0.25, 'right': 0.25, 'down': 0.25, 'left': 0.25}],
 [{'up': 0.25, 'right': 0.25, 'down': 0.25, 'left': 0.25},
  {'up': 0.25, 'right': 0.25, 'down': 0.25, 'left': 0.25},
  {'up': 0.25, 'right': 0.25, 'down': 0.25, 'left': 0.25},
  {'up': 0.25, 'right': 0.25, 'down': 0.25, 'left': 0.25}]]

In [558]:
policy_iteration(
    default_grid,
    policy=uniform_policy,
);

---------- policy_iteration ------------
policy_evaluation converged in 80 iterations
  r/c   0    1    2    3 
   0 0.15 0.27 0.51    0 
   1  0.1    0 0.37 0.52 
   2  0.1 0.14 0.24 0.31 
------------------
actions changed = 6
r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  →  ↑ 
 2  →  →  ↑  ↑ 
------------------
policy_evaluation converged in 6 iterations
  r/c   0    1    2    3 
   0 0.81  0.9  1.0    0 
   1 0.73    0  0.9  1.0 
   2 0.66 0.73 0.81  0.9 
------------------
actions changed = 2
r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  ↑  ↑ 
 2  ↑  →  ↑  ↑ 
------------------
policy_evaluation converged in 6 iterations
  r/c   0    1    2    3 
   0 0.81  0.9  1.0    0 
   1 0.73    0  0.9  1.0 
   2 0.66 0.73 0.81  0.9 
------------------
actions changed = 0
r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  ↑  ↑ 
 2  ↑  →  ↑  ↑ 
------------------
policy_iteration converged in 3 iterations


In [559]:
policy_iteration(
    default_grid,
    policy=uniform_policy,
    pass_incumbent_policy=True
);

---------- policy_iteration ------------
policy_evaluation converged in 80 iterations
  r/c   0    1    2    3 
   0 0.15 0.27 0.51    0 
   1  0.1    0 0.37 0.52 
   2  0.1 0.14 0.24 0.31 
------------------
actions changed = 6
r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  →  ↑ 
 2  →  →  ↑  ↑ 
------------------
policy_evaluation converged in 6 iterations
  r/c   0    1    2    3 
   0 0.81  0.9  1.0    0 
   1 0.73    0  0.9  1.0 
   2 0.66 0.73 0.81  0.9 
------------------
actions changed = 0
r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  →  ↑ 
 2  →  →  ↑  ↑ 
------------------
policy_iteration converged in 2 iterations


## Experiments

## ε-greedy evaluation next to a pit 

In [560]:
grid = Grid(
    rows=3,
    cols=4,
    step_reward=-0.04,
    terminals={(0, 3): 1, (1, 3): -1},
    walls={(1, 1)},
)

In [561]:
grid.render()

  ·   ·   ·   1 
  ·   #   ·  -1 
  ·   ·   ·   · 


In [562]:
pi_star, V, converged = value_iteration(grid)

---------- value_iteration ------------
  r/c   0    1    2    3 
   0 -0.04 -0.04  1.0    0 
   1 -0.04    0 -0.04    0 
   2 -0.04 -0.04 -0.04 -0.04 
------------------
  r/c   0    1    2    3 
   0 -0.08 0.86  1.0    0 
   1 -0.08    0 0.86    0 
   2 -0.08 -0.08 -0.08 -0.08 
------------------
  r/c   0    1    2    3 
   0 0.73 0.86  1.0    0 
   1 -0.11    0 0.86    0 
   2 -0.11 -0.11 0.73 -0.11 
------------------
  r/c   0    1    2    3 
   0 0.73 0.86  1.0    0 
   1 0.62    0 0.86    0 
   2 -0.14 0.62 0.73 0.62 
------------------
  r/c   0    1    2    3 
   0 0.73 0.86  1.0    0 
   1 0.62    0 0.86    0 
   2 0.52 0.62 0.73 0.62 
------------------
  r/c   0    1    2    3 
   0 0.73 0.86  1.0    0 
   1 0.62    0 0.86    0 
   2 0.52 0.62 0.73 0.62 
------------------
r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  ↑  -1 
 2  ↑  →  ↑  ← 
------------------
value_iteration converged in 6 iterations


In [563]:
pi_star

[[{'right': 1.0}, {'right': 1.0}, {'right': 1.0}, None],
 [{'up': 1.0}, None, {'up': 1.0}, None],
 [{'up': 1.0}, {'right': 1.0}, {'up': 1.0}, {'left': 1.0}]]

In [564]:
def eps_soft(grid: Grid, policy, eps=0.1):
    policy = [row[:] for row in policy]
    action_keys = list(grid.actions.keys())
    n_actions = len(action_keys)
    for r in range(grid.rows):
        for c in range(grid.cols):
            if policy[r][c] is None:
                continue
            greedy_action = max(policy[r][c], key=policy[r][c].get)
            policy[r][c] = {
                action: eps / n_actions
                for action in action_keys
                if action != greedy_action
            }
            policy[r][c][greedy_action] = 1.0 - eps + eps / n_actions
    return policy

In [565]:
new_policy = eps_soft(grid, pi_star, eps=0.9)
new_policy

[[{'up': 0.225, 'down': 0.225, 'left': 0.225, 'right': 0.32499999999999996},
  {'up': 0.225, 'down': 0.225, 'left': 0.225, 'right': 0.32499999999999996},
  {'up': 0.225, 'down': 0.225, 'left': 0.225, 'right': 0.32499999999999996},
  None],
 [{'right': 0.225, 'down': 0.225, 'left': 0.225, 'up': 0.32499999999999996},
  None,
  {'right': 0.225, 'down': 0.225, 'left': 0.225, 'up': 0.32499999999999996},
  None],
 [{'right': 0.225, 'down': 0.225, 'left': 0.225, 'up': 0.32499999999999996},
  {'up': 0.225, 'down': 0.225, 'left': 0.225, 'right': 0.32499999999999996},
  {'right': 0.225, 'down': 0.225, 'left': 0.225, 'up': 0.32499999999999996},
  {'up': 0.225, 'right': 0.225, 'down': 0.225, 'left': 0.32499999999999996}]]

In [566]:
sum(new_policy[0][0].values())

1.0

In [567]:
V, converged = policy_evaluation(grid, eps_soft(grid, pi_star, eps=0.9));

policy_evaluation converged in 65 iterations


In [568]:
show_V(grid, V)

  r/c   0    1    2    3 
   0 -0.13 0.04  0.3    0 
   1 -0.24    0 -0.32    0 
   2 -0.32 -0.38 -0.43 -0.64 
------------------


In [569]:
policy = read_policy(grid, V)
policy

[[{'right': 1.0}, {'right': 1.0}, {'right': 1.0}, None],
 [{'up': 1.0}, None, {'up': 1.0}, None],
 [{'up': 1.0}, {'left': 1.0}, {'up': 1.0}, {'left': 1.0}]]

In [570]:
render_policy(grid, pi_star)

r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  ↑  -1 
 2  ↑  →  ↑  ← 
------------------


In [571]:
render_policy(grid, policy)

r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  ↑  -1 
 2  ↑  ←  ↑  ← 
------------------


In [572]:
for eps in [0, 0.05, 0.1, 0.2, 0.5, 0.9, 1.0]:
    print(f'{eps=}')
    V, converged = policy_evaluation(grid, eps_soft(grid, pi_star, eps=eps));
    show_V(grid, V)
    policy = read_policy(grid, V)
    render_policy(grid, policy)

eps=0
policy_evaluation converged in 6 iterations
  r/c   0    1    2    3 
   0 0.73 0.86  1.0    0 
   1 0.62    0 0.86    0 
   2 0.52 0.62 0.73 0.62 
------------------
r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  ↑  -1 
 2  ↑  →  ↑  ← 
------------------
eps=0.05
policy_evaluation converged in 14 iterations
  r/c   0    1    2    3 
   0 0.72 0.85 0.99    0 
   1  0.6    0 0.82    0 
   2  0.5 0.58 0.69 0.56 
------------------
r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  ↑  -1 
 2  ↑  →  ↑  ← 
------------------
eps=0.1
policy_evaluation converged in 17 iterations
  r/c   0    1    2    3 
   0 0.69 0.83 0.98    0 
   1 0.57    0 0.78    0 
   2 0.47 0.54 0.65  0.5 
------------------
r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  ↑  -1 
 2  ↑  →  ↑  ← 
------------------
eps=0.2
policy_evaluation converged in 21 iterations
  r/c   0    1    2    3 
   0 0.65 0.79 0.95    0 
   1 0.52    0  0.7    0 
   2 0.41 0.44 0.56 0.37 
------------------
r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  ↑  -

Fair — let me rebuild it from the bottom.

What V is measuring in each row
-------------------------------

Each row of that table is the answer to a _different_ question. Not "how well do I estimate V\*" — every row is an exact answer, converged to 1e-6. The question changes:

*   **ε=0**: "If I always take π\*'s action, what do I get?" → that's V\*.
*   **ε=0.5**: "If I take π\*'s action only 62.5% of the time and a random one otherwise, what do I get?"

At ε=0.5 with 4 actions, the greedy action's probability is `0.5 + 0.5/4 = 0.625`, and each of the other three gets `0.125`. So the ε=0.5 row is the true expected return of a genuinely clumsy agent — one that stumbles more than a third of the time, every step, forever. Not a noisy estimate of a good agent.

Stumbling costs you two different things
----------------------------------------

**Cost 1 — wasted steps.** A wrong action bumps a wall or walks backward. You pay `-0.04` and try again. Every cell suffers this equally, because wandering is wandering. That's why the far column drops in lockstep: −0.86, −0.86, −0.84.

**Cost 2 — falling in the pit.** This only exists if a wrong action can land you on (1,3). It doesn't cost you a step — it ends the episode at −1. You never reach the +1 at all.

So, roughly:

    V^eps(s)  =  V*(s)  -  (wandering tax, everywhere)  -  (pit risk, only near the pit)
    

At ε=0.9 the far column loses ~0.85 and the pit cells lose ~1.2. The extra ~0.35 is cost 2.

Why (2,3) specifically
----------------------

(2,3) sits directly below the pit:

     ·  ·  ·  +1
     ·  #  ·  -1     <- pit at (1,3)
     ·  ·  ·   ·     <- (2,3) is here, right underneath
    

π\* tells it to go `left`. But under ε-greedy, `up` gets probability `ε/4` — at ε=0.5 that's **12.5% per visit**, and `up` from (2,3) is the pit. So on average one visit in eight ends the episode at −1.

Contrast (2,0) in the far column. Its wrong actions bump the left wall or step into empty cells. Cost: `-0.04` and some lost time. Nothing catastrophic is reachable.

Same amount of randomness. Wildly different consequence.

The crossover, restated
-----------------------

    eps=0     (2,3)=0.62   (2,0)=0.52     (2,3) wins
    eps=0.2   (2,3)=0.37   (2,0)=0.41     (2,0) wins
    

At ε=0, (2,3) is the better place to be — it's 2 steps from the goal, (2,0) is 4. That head start is worth about 0.10.

As ε rises, both lose to wandering, but (2,3) _additionally_ leaks value into the pit. Once that leak exceeds the 0.10 head start — somewhere around ε≈0.15 — being closer to the goal stops being worth it. **A clumsy agent is better off far away in safe territory than close to the reward but next to a hazard.**

Why this is the SARSA/Q-learning split
--------------------------------------

Those are two different rows of your table:

*   **Q-learning** learns V\* — the ε=0 row. It answers "what's this worth assuming I act perfectly from here." It says (2,3) is worth 0.62.
*   **SARSA** learns V^ε-greedy — the ε=0.5 row. It answers "what's this worth given I'll keep exploring." It says (2,3) is worth −0.07.

SARSA's numbers _include_ the cost of its own clumsiness, so it routes away from the pit while learning. Q-learning's don't, so it takes the optimal-if-perfect path and falls in repeatedly during training. Same environment, same ε — different question, different answer.

Your table is that difference, computed exactly, before either algorithm exists.